# Random Forest v2.2 — Memory-Optimised Build

## Memory Optimisations Applied
| Technique | Before | After | Memory Saving |
|---|---|---|---|
| `n_estimators` | 500 | **200** | ~60% less RAM |
| `n_jobs` | -1 (all cores) | **2** | Limits parallel tree RAM |
| `max_depth` | 20 | **15** | Shallower trees = less RAM |
| `max_samples` | 100% | **0.6** | Each tree uses 60% of data |
| Dataset sampling | 100% | **60%** | Loads only 60% of CSV |
| `gc.collect()` | No | **Yes** | Frees memory after heavy steps |

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pickle
import gc
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_auc_score, fbeta_score
)
import warnings
warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
plt.style.use('ggplot')
print('[OK] Libraries loaded')

## 1. Load Data (60% Sample to Save RAM)

In [ ]:
FILE_PATH = r"C:\Users\shenal\Downloads\reseraach\PCAPS_Used\Final_Balanced_Attack_and_Benign\Final_balanced_Attack_and_Benign_new_Shuffled.csv"

# MEMORY SAVE: Sample 60% of the dataset for training
# RF generalises well even on 60% — sufficient for research
SAMPLE_FRACTION = 0.6   # << Reduce to 0.4 if still out of memory

print(f'[INFO] Loading {SAMPLE_FRACTION*100:.0f}% sample of: {FILE_PATH}')
df = pd.read_csv(FILE_PATH)
print(f'[INFO] Full size: {len(df):,} rows')

# Stratified sample — keeps class balance intact
df = df.groupby('label', group_keys=False).apply(
    lambda x: x.sample(frac=SAMPLE_FRACTION, random_state=42)
).reset_index(drop=True)
print(f'[OK]  Sampled: {len(df):,} rows')
print(df['label'].value_counts())

# Free original
gc.collect()

## 2. Preprocessing + Log-Transform

In [ ]:
COLS_TO_DROP = ['src_ip', 'dst_ip', 'src_port', 'dst_port', 'protocol_number']
df = df.drop(columns=COLS_TO_DROP, errors='ignore')
df.replace([np.inf, -np.inf], 0, inplace=True)
df.fillna(0, inplace=True)

LABEL_MAP = {'BENIGN': 0, 'ATTACK': 1}
df['label'] = df['label'].str.upper().map(LABEL_MAP).fillna(0).astype(int)
print(f'Labels: {dict(df["label"].value_counts())}')

PROTOCOL_CLASSES = ['DOH', 'DOT', 'TRADITIONAL', 'UNKNOWN', 'TCP', 'UDP']
le_proto = LabelEncoder()
le_proto.fit(PROTOCOL_CLASSES)
df['protocol'] = df['protocol'].astype(str).apply(
    lambda x: x if x in PROTOCOL_CLASSES else 'UNKNOWN'
)
df['protocol'] = le_proto.transform(df['protocol'])

# Same selective log-transform as XGBoost v2.3
LOG_FEATURES = [
    'bwd_packets_per_sec',
    'flow_bytes_per_sec',
    'flow_packets_per_sec',
    'fwd_packets_per_sec',
    'total_fwd_packets',
    'total_bwd_packets',
    # dns_queries_per_second and dns_amplification_factor kept RAW
]
for col in LOG_FEATURES:
    if col in df.columns:
        df[col] = np.log1p(df[col].clip(lower=0))

# Downcast floats to float32 — cuts memory by ~50%
float_cols = df.select_dtypes(include='float64').columns
df[float_cols] = df[float_cols].astype(np.float32)
int_cols = df.select_dtypes(include='int64').columns
df[int_cols] = df[int_cols].astype(np.int32)
print(f'[OK] Downcasted to float32/int32 — memory usage: {df.memory_usage(deep=True).sum()/1e6:.1f} MB')
gc.collect()

## 3. Train / Test Split

In [ ]:
X = df.drop(columns=['label'])
y = df['label']
del df; gc.collect()   # Free full dataframe immediately after split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
del X, y; gc.collect()  # Free unsplit data

print(f'Train: {X_train.shape}  ({X_train.memory_usage(deep=True).sum()/1e6:.0f} MB)')
print(f'Test:  {X_test.shape}  ({X_test.memory_usage(deep=True).sum()/1e6:.0f} MB)')

## 4. Random Forest Training — Memory Safe

**If your computer still gets stuck:**
- Reduce `n_estimators` to `100`
- Reduce `SAMPLE_FRACTION` to `0.4` in Cell 2  
- Set `n_jobs=1` to use only one core

In [ ]:
model = RandomForestClassifier(
    n_estimators      = 200,          # 500→200: biggest memory reduction
    max_depth         = 15,           # 20→15: shallower = less RAM per tree
    min_samples_leaf  = 10,           # 5→10: fewer nodes per tree
    min_samples_split = 20,
    max_features      = 'sqrt',       # Only ~6 features per split
    max_samples       = 0.6,          # Each tree uses 60% of training rows
    class_weight      = 'balanced',
    n_jobs            = 2,            # -1→2: limits parallel RAM usage
    random_state      = 42,
    verbose           = 1,
)

print('[TRAIN] Memory-safe RF: 200 trees, max_depth=15, n_jobs=2, max_samples=0.6')
model.fit(X_train, y_train)
print('[DONE] Training complete.')
gc.collect()

## 5. Threshold Tuning (beta=3.0 — Attack Recall Priority)

In [ ]:
y_prob = model.predict_proba(X_test)[:, 1]
BETA = 3.0

thresholds = np.arange(0.05, 0.95, 0.01)
best_thresh, best_score = 0.5, 0
for t in thresholds:
    score = fbeta_score(y_test, (y_prob >= t).astype(int), beta=BETA, zero_division=0)
    if score > best_score:
        best_score, best_thresh = score, t

print(f'[TUNING] Optimal threshold = {best_thresh:.2f}  (F-{BETA} = {best_score:.4f})')

scores = [fbeta_score(y_test, (y_prob >= t).astype(int), beta=BETA, zero_division=0)
          for t in thresholds]
plt.figure(figsize=(10, 4))
plt.plot(thresholds, scores, color='forestgreen')
plt.axvline(best_thresh, color='red',  linestyle='--', label=f'Best={best_thresh:.2f}')
plt.axvline(0.5,         color='gray', linestyle=':',  label='Default=0.50')
plt.xlabel('Threshold'); plt.ylabel(f'F-beta (beta={BETA})')
plt.title('RF Threshold Tuning'); plt.legend(); plt.show()

## 6. Balance Report

In [ ]:
y_pred = (y_prob >= best_thresh).astype(int)

print(f'=== CLASSIFICATION REPORT (threshold={best_thresh:.2f}) ===')
print(classification_report(y_test, y_pred, target_names=['BENIGN','ATTACK'], digits=4))

auc = roc_auc_score(y_test, y_prob)
print(f'ROC-AUC: {auc:.4f}')

cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Greens',
            xticklabels=['BENIGN','ATTACK'], yticklabels=['BENIGN','ATTACK'])
plt.title(f'RF Confusion Matrix @ threshold={best_thresh:.2f}')
plt.ylabel('Actual'); plt.xlabel('Predicted')
plt.tight_layout(); plt.show()

tn, fp, fn, tp = cm.ravel()
benign_recall  = tn / (tn + fp) if (tn + fp) > 0 else 0
attack_recall  = tp / (tp + fn) if (tp + fn) > 0 else 0
false_pos_rate = fp / (fp + tn) if (fp + tn) > 0 else 0

print(f'\n===============================')
print(f' RF BALANCE REPORT')
print(f'===============================')
print(f' BENIGN Recall  : {benign_recall*100:6.1f}%  (target: >90%)')
print(f' ATTACK Recall  : {attack_recall*100:6.1f}%  (target: >90%)')
print(f' False Pos Rate : {false_pos_rate*100:6.1f}%  (target: <10%)')
print(f' ROC-AUC        : {auc:.4f}')
print(f'===============================')

if attack_recall < 0.85:
    print('\n[ADVICE] Attack recall low. Try: class_weight={0:1, 1:3}')
elif benign_recall < 0.85:
    print('\n[ADVICE] Too many FP. Try: increase min_samples_leaf to 20')
else:
    print('\n[GOOD] Both recalls above 85%!')

## 7. Feature Importance

In [ ]:
importance = pd.Series(model.feature_importances_, index=X_train.columns)
top20 = importance.nlargest(20)

plt.figure(figsize=(10, 7))
top20.sort_values().plot(kind='barh', color='forestgreen')
plt.title('Top 20 Feature Importances — Random Forest v2.2')
plt.xlabel('Importance Score'); plt.tight_layout(); plt.show()

print('\n--- Top 10 Features ---')
for feat, imp in top20.head(10).items():
    bar = '#' * int(imp * 100)
    print(f'  {feat:<35} {imp:.4f}  {bar}')

## 8. Save Model Bundle

In [ ]:
bundle = {
    'model':            model,
    'threshold':        best_thresh,
    'log_features':     LOG_FEATURES,
    'protocol_classes': PROTOCOL_CLASSES,
    'model_type':       'random_forest',
    'beta':             BETA,
    'sample_fraction':  SAMPLE_FRACTION,
}

with open('rf_model_v2.2.pkl', 'wb') as f:
    pickle.dump(bundle, f)

print(f'[SAVED] rf_model_v2.2.pkl')
print(f'  threshold      = {best_thresh:.2f}')
print(f'  ATTACK Recall  = {attack_recall*100:.1f}%')
print(f'  BENIGN Recall  = {benign_recall*100:.1f}%')
print(f'  ROC-AUC        = {auc:.4f}')